# 05 — Pre-deployment Model Comparison

Notebook này tổng hợp **toàn bộ kết quả trước khi deploy Android**:

- 4 model `.pt` sau training;
- 4 model `.tflite` sau export LiteRT;
- mức thay đổi metric sau conversion;
- ảnh hưởng của `n/s` và `320/640`;
- trade-off sơ bộ giữa accuracy và model size.

Notebook này **chưa phải báo cáo cuối**. Sau khi benchmark trên Android, ta sẽ có notebook riêng cho latency, FPS, RAM, CPU/GPU và thermal rồi mới ghép thành kết luận cuối.


## 1. Dữ liệu kết quả hiện tại

Các giá trị `.pt` lấy từ kết quả test sau training.  
Các giá trị `.tflite` lấy từ full validation trên cùng test set 1,151 ảnh.

Dữ liệu hiện tại:
- `yolo11n_320`
- `yolo11n_640`
- `yolo11s_320`
- `yolo11s_640`


In [ ]:
import pandas as pd
from pathlib import Path

# Metric .pt sau training
pt_results = {
    "yolo11n_320": {"precision": 0.766237, "recall": 0.709237, "mAP50": 0.741643, "mAP50_95": 0.577623},
    "yolo11n_640": {"precision": 0.922132, "recall": 0.843105, "mAP50": 0.929526, "mAP50_95": 0.756673},
    "yolo11s_320": {"precision": 0.917825, "recall": 0.851263, "mAP50": 0.901483, "mAP50_95": 0.709645},
    "yolo11s_640": {"precision": 0.950797, "recall": 0.924849, "mAP50": 0.969630, "mAP50_95": 0.804516},
}

# Metric .tflite sau full validation
tflite_results = {
    "yolo11n_320": {"precision": 0.780523, "recall": 0.660411, "mAP50": 0.729657, "mAP50_95": 0.577848},
    "yolo11n_640": {"precision": 0.913202, "recall": 0.832230, "mAP50": 0.920290, "mAP50_95": 0.749624},
    "yolo11s_320": {"precision": 0.914394, "recall": 0.846487, "mAP50": 0.897511, "mAP50_95": 0.708333},
    "yolo11s_640": {"precision": 0.935172, "recall": 0.942058, "mAP50": 0.970428, "mAP50_95": 0.806390},
}

# Kích thước TFLite sau export (MiB)
tflite_size_mib = {
    "yolo11n_320": 10.166,
    "yolo11n_640": 10.178,
    "yolo11s_320": 36.206,
    "yolo11s_640": 36.278,
}

rows = []

for model_name in pt_results:
    row = {
        "experiment": model_name,
        "imgsz": 320 if "320" in model_name else 640,
        "variant": "n" if "11n" in model_name else "s",
        "tflite_size_mib": tflite_size_mib[model_name],
    }

    for metric in ["precision", "recall", "mAP50", "mAP50_95"]:
        row[f"pt_{metric}"] = pt_results[model_name][metric]
        row[f"tflite_{metric}"] = tflite_results[model_name][metric]
        row[f"delta_{metric}"] = tflite_results[model_name][metric] - pt_results[model_name][metric]

    rows.append(row)

df = pd.DataFrame(rows)
display(df.round(6))


## 2. So sánh mAP50: `.pt` vs `.tflite`

Biểu đồ này cho thấy conversion LiteRT có làm thay đổi accuracy tổng thể đáng kể hay không.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

x = np.arange(len(df))
width = 0.36

plt.figure(figsize=(9, 5))
plt.bar(x - width/2, df["pt_mAP50"], width, label=".pt")
plt.bar(x + width/2, df["tflite_mAP50"], width, label=".tflite")
plt.xticks(x, df["experiment"], rotation=20)
plt.ylabel("mAP50")
plt.title("mAP50: PyTorch vs LiteRT/TFLite")
plt.ylim(0, 1.05)
plt.legend()
plt.tight_layout()
plt.show()


## 3. So sánh mAP50-95: `.pt` vs `.tflite`

`mAP50-95` khắt khe hơn về localization nên là metric chính để theo dõi ảnh hưởng của conversion.


In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(x - width/2, df["pt_mAP50_95"], width, label=".pt")
plt.bar(x + width/2, df["tflite_mAP50_95"], width, label=".tflite")
plt.xticks(x, df["experiment"], rotation=20)
plt.ylabel("mAP50-95")
plt.title("mAP50-95: PyTorch vs LiteRT/TFLite")
plt.ylim(0, 1.0)
plt.legend()
plt.tight_layout()
plt.show()


## 4. Mức thay đổi mAP50-95 sau conversion

Giá trị gần `0` nghĩa là conversion gần như giữ nguyên chất lượng.  
Giá trị âm là giảm, dương là tăng nhẹ.


In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(df["experiment"], df["delta_mAP50_95"])
plt.axhline(0)
plt.ylabel("Δ mAP50-95 (TFLite - PT)")
plt.title("Thay đổi mAP50-95 sau export LiteRT")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## 5. Precision và Recall sau conversion

Conversion có thể làm Precision và Recall dịch chuyển theo hai hướng khác nhau dù mAP gần như giữ nguyên. Vì vậy không nên chỉ nhìn một metric riêng lẻ.


In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(x - width/2, df["pt_precision"], width, label="PT Precision")
plt.bar(x + width/2, df["tflite_precision"], width, label="TFLite Precision")
plt.xticks(x, df["experiment"], rotation=20)
plt.ylabel("Precision")
plt.title("Precision: PyTorch vs LiteRT/TFLite")
plt.ylim(0, 1.05)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(x - width/2, df["pt_recall"], width, label="PT Recall")
plt.bar(x + width/2, df["tflite_recall"], width, label="TFLite Recall")
plt.xticks(x, df["experiment"], rotation=20)
plt.ylabel("Recall")
plt.title("Recall: PyTorch vs LiteRT/TFLite")
plt.ylim(0, 1.05)
plt.legend()
plt.tight_layout()
plt.show()


## 6. Trade-off sơ bộ: TFLite model size vs mAP50-95

Đây mới chỉ là **pre-deployment trade-off**.  
Sau Android benchmark, trục hiệu năng quan trọng hơn sẽ là latency/FPS thay vì chỉ model size.


In [ ]:
plt.figure(figsize=(8, 5))

for _, row in df.iterrows():
    plt.scatter(row["tflite_size_mib"], row["tflite_mAP50_95"], s=80)
    plt.annotate(
        row["experiment"],
        (row["tflite_size_mib"], row["tflite_mAP50_95"]),
        xytext=(5, 5),
        textcoords="offset points",
    )

plt.xlabel("TFLite model size (MiB)")
plt.ylabel("TFLite mAP50-95")
plt.title("Pre-deployment trade-off: Model Size vs Accuracy")
plt.tight_layout()
plt.show()


## 7. Ảnh hưởng của input size 320 vs 640

So sánh này giúp đánh giá mức độ quan trọng của resolution với bài toán traffic sign detection.


In [ ]:
resolution_df = df[["experiment", "variant", "imgsz", "tflite_mAP50_95"]].copy()
display(resolution_df.sort_values(["variant", "imgsz"]))


In [ ]:
plt.figure(figsize=(8, 5))

for variant in ["n", "s"]:
    part = resolution_df[resolution_df["variant"] == variant].sort_values("imgsz")
    plt.plot(part["imgsz"], part["tflite_mAP50_95"], marker="o", label=f"YOLO11{variant}")

plt.xlabel("Input size")
plt.ylabel("TFLite mAP50-95")
plt.title("Ảnh hưởng của input resolution")
plt.xticks([320, 640])
plt.legend()
plt.tight_layout()
plt.show()


## 8. Nhận xét tự động từ kết quả hiện tại

Các nhận xét dưới đây chỉ dùng cho giai đoạn **trước Android deployment**.


In [ ]:
best_accuracy = df.loc[df["tflite_mAP50_95"].idxmax()]
smallest_model = df.loc[df["tflite_size_mib"].idxmin()]
best_n = df[df["variant"] == "n"].sort_values("tflite_mAP50_95", ascending=False).iloc[0]

print("===== NHẬN XÉT PRE-DEPLOYMENT =====")
print()

print(
    f"1. Accuracy cao nhất: {best_accuracy['experiment']} "
    f"(TFLite mAP50-95 = {best_accuracy['tflite_mAP50_95']:.4f})."
)

print(
    f"2. Model TFLite nhỏ nhất: {smallest_model['experiment']} "
    f"({smallest_model['tflite_size_mib']:.3f} MiB)."
)

print(
    f"3. Candidate nano tốt nhất: {best_n['experiment']} "
    f"(mAP50-95 = {best_n['tflite_mAP50_95']:.4f}, "
    f"size = {best_n['tflite_size_mib']:.3f} MiB)."
)

print(
    "4. n-640 vượt n-320 rõ rệt, cho thấy resolution 640 quan trọng với traffic sign/small-object detection."
)

print(
    "5. s-640 đạt accuracy cao nhất nhưng file TFLite lớn hơn n-640 khoảng "
    f"{df.loc[df['experiment']=='yolo11s_640','tflite_size_mib'].iloc[0] / df.loc[df['experiment']=='yolo11n_640','tflite_size_mib'].iloc[0]:.2f} lần."
)

print(
    "6. Conversion PT → TFLite chỉ làm mAP50-95 thay đổi rất nhỏ ở cả 4 model, "
    "nên artifact deployment hiện không cho thấy suy giảm accuracy nghiêm trọng."
)


## 9. Lưu artifact phân tích

Notebook lưu:
- bảng CSV tổng hợp;
- các số liệu để dùng lại khi viết báo cáo.

Sau Android benchmark, ta sẽ ghép thêm:
- inference latency;
- P50/P95;
- processed FPS;
- RAM;
- CPU/GPU;
- thermal;
- model size;
- accuracy;

để chọn model cuối bằng accuracy-performance trade-off / Pareto analysis.


In [ ]:
# Tự tìm project root
candidate_roots = [
    Path.cwd().resolve(),
    Path.cwd().resolve().parent,
    Path(r"D:\Project\TrafficSignAI"),
]

PROJECT_ROOT = next(
    (p for p in candidate_roots if (p / "models").exists() and (p / "VR-TSD-2").exists()),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Không tìm thấy project TrafficSignAI.")

OUTPUT_DIR = PROJECT_ROOT / "runs" / "predeploy_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CSV_PATH = OUTPUT_DIR / "predeploy_model_comparison.csv"
df.to_csv(CSV_PATH, index=False)

print("Saved:", CSV_PATH)


## Kết luận giai đoạn pre-deployment

Tại thời điểm này:

- `YOLO11s-640` là model có accuracy cao nhất.
- `YOLO11n-640` là candidate mobile đáng chú ý vì accuracy tốt nhưng model nhỏ hơn rất nhiều.
- `YOLO11n-320` giảm accuracy mạnh khi giảm resolution.
- `YOLO11s-320` không vượt `YOLO11n-640` về mAP50-95 dù model lớn hơn nhiều.
- PT → TFLite conversion nhìn chung giữ accuracy ổn định.

**Chưa chọn model cuối.**  
Quyết định cuối chỉ nên thực hiện sau khi có Android latency/FPS/RAM/CPU/thermal.
